In [27]:
import pandas as pd
import json
import re
from pathlib import Path

## 1. Reading Datasets

In [28]:
BASE_DIR = Path.cwd().parent 
data_dir = BASE_DIR / "data"

train_df = pd.read_parquet(data_dir / "train-00000-of-00001.parquet")
dev_df   = pd.read_parquet(data_dir / "dev-00000-of-00001.parquet")
test_df  = pd.read_parquet(data_dir / "test-00000-of-00001.parquet")

## 2. Text Normalization and Cleaning

In [29]:
def normalize_text(text):
    if not isinstance(text, str):
        return text
    
    text = re.sub(r'\s+', ' ', text).strip()
    
    text = re.sub(r'http\S+|www\S+', '', text)
    
    text = re.sub(r'\S+@\S+', '', text)
    
    text = re.sub(r'[^\w\s\.\,\!\?\'\"]', '', text)
    
    return text.strip()

## 3. Schema Validation

In [30]:
def validate_output_schema(output_json):
    try:
        if "aspects" not in output_json:
            return False, "Missing required field: 'aspects'"
        
        if not isinstance(output_json["aspects"], list):
            return False, "Field 'aspects' must be a list"
        
        for i, aspect in enumerate(output_json["aspects"]):
            if not isinstance(aspect, dict):
                return False, f"Aspect {i} is not a dictionary"

            if "term" not in aspect or "polarity" not in aspect:
                return False, f"Aspect {i} missing required fields. Expected: 'term' and 'polarity'"

            if not isinstance(aspect["term"], str):
                return False, f"Aspect {i}: 'term' must be a string"
            if not isinstance(aspect["polarity"], str):
                return False, f"Aspect {i}: 'polarity' must be a string"
        
        return True, "Valid"
    
    except Exception as e:
        return False, str(e)

test_valid_json = {
    "aspects": [
        {"term": "cord", "polarity": "neutral"},
        {"term": "battery life", "polarity": "positive"}
    ]
}

test_invalid_json = {
    "aspects": [
        {"term": "quality"}
    ]
}

test_invalid_json_2 = {
    "missing_aspects": "field"
}

print("Valid schema:", validate_output_schema(test_valid_json))
print("Invalid schema (missing polarity):", validate_output_schema(test_invalid_json))
print("Invalid schema (missing aspects):", validate_output_schema(test_invalid_json_2))


Valid schema: (True, 'Valid')
Invalid schema (missing polarity): (False, "Aspect 0 missing required fields. Expected: 'term' and 'polarity'")
Invalid schema (missing aspects): (False, "Missing required field: 'aspects'")


## 4. Data Cleaning and Filtering

In [31]:
def clean_dataset(df, dataset_name="dataset"):
    report = {
        "name": dataset_name,
        "original_rows": len(df),
        "duplicates_removed": 0,
        "invalid_json_removed": 0,
        "invalid_schema_removed": 0,
        "final_rows": 0
    }
    
    df_cleaned = df.drop_duplicates().reset_index(drop=True)
    report["duplicates_removed"] = len(df) - len(df_cleaned)
    
    valid_rows = []
    invalid_rows_list = []
    
    for idx, row in df_cleaned.iterrows():
        try:
            output_json = json.loads(row["output"])
            is_valid, msg = validate_output_schema(output_json)
            
            if is_valid:
                if "input" in row and isinstance(row["input"], str):
                    row["input"] = normalize_text(row["input"])
                valid_rows.append(row)
            else:
                invalid_rows_list.append((idx, "Invalid schema", msg))
                report["invalid_schema_removed"] += 1
        except json.JSONDecodeError as e:
            invalid_rows_list.append((idx, "Invalid JSON", str(e)))
            report["invalid_json_removed"] += 1
    
    df_cleaned = pd.DataFrame(valid_rows).reset_index(drop=True)
    report["final_rows"] = len(df_cleaned)
    report["invalid_rows_detail"] = invalid_rows_list[:10]
    
    return df_cleaned, report

print("=" * 60)
print("CLEANING DATASETS")
print("=" * 60 + "\n")

train_cleaned, train_report = clean_dataset(train_df, "Train")
dev_cleaned, dev_report = clean_dataset(dev_df, "Dev")
test_cleaned, test_report = clean_dataset(test_df, "Test")

for report in [train_report, dev_report, test_report]:
    print(f"\n{report['name']} Dataset Report:")
    print(f"  Original rows: {report['original_rows']}")
    print(f"  Duplicates removed: {report['duplicates_removed']}")
    print(f"  Invalid JSON removed: {report['invalid_json_removed']}")
    print(f"  Invalid schema removed: {report['invalid_schema_removed']}")
    print(f"  Final rows: {report['final_rows']}")
    if report['original_rows'] > 0:
        print(f"  Retention rate: {report['final_rows'] / report['original_rows'] * 100:.2f}%")


CLEANING DATASETS


Train Dataset Report:
  Original rows: 5959
  Duplicates removed: 0
  Invalid JSON removed: 0
  Invalid schema removed: 0
  Final rows: 5959
  Retention rate: 100.00%

Dev Dataset Report:
  Original rows: 200
  Duplicates removed: 0
  Invalid JSON removed: 0
  Invalid schema removed: 0
  Final rows: 200
  Retention rate: 100.00%

Test Dataset Report:
  Original rows: 1572
  Duplicates removed: 0
  Invalid JSON removed: 0
  Invalid schema removed: 0
  Final rows: 1572
  Retention rate: 100.00%


## 5. Convert to Instruction-Style Format for LoRA Training

In [32]:
def convert_to_instruction_format(df):

    instruction_data = []
    
    for idx, row in df.iterrows():
        try:
            input_text = row["input"]
            instruction = row["instruction"]
            response = row["output"]
            
            instruction_data.append({
                "input": input_text,
                "instruction": instruction,
                "output": response
            })
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            continue
    
    return instruction_data

train_instruction = convert_to_instruction_format(train_cleaned)
dev_instruction = convert_to_instruction_format(dev_cleaned)
test_instruction = convert_to_instruction_format(test_cleaned)

print(f"Train instruction pairs: {len(train_instruction)}")
print(f"Dev instruction pairs: {len(dev_instruction)}")
print(f"Test instruction pairs: {len(test_instruction)}")

print("\nSample instruction-response pair:")
if len(train_instruction) > 0:
    print(json.dumps(train_instruction[0], indent=2)[:500] + "...")
else:
    print("No instruction pairs created. Check data cleaning results above.")


Train instruction pairs: 5959
Dev instruction pairs: 200
Test instruction pairs: 1572

Sample instruction-response pair:
{
  "input": "I charge it at night and skip taking the cord with me because of the good battery life.",
  "instruction": "Extract the aspect terms and their polarity from the text.",
  "output": "{\"aspects\": [{\"term\": \"cord\", \"polarity\": \"neutral\"}, {\"term\": \"battery life\", \"polarity\": \"positive\"}]}"
}...


## 6. Save Processed Data

In [33]:
BASE_DIR = Path.cwd().parent 
processed_dir = BASE_DIR / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print("Saving cleaned datasets...")
train_cleaned.to_parquet(processed_dir / "train_cleaned.parquet")
dev_cleaned.to_parquet(processed_dir / "dev_cleaned.parquet")
test_cleaned.to_parquet(processed_dir / "test_cleaned.parquet")
print("Cleaned data saved as parquet files")

print("\nSaving instruction-response format for LoRA training...")
with open(processed_dir / "train_instructions.jsonl", "w") as f:
    for item in train_instruction:
        f.write(json.dumps(item) + "\n")

with open(processed_dir / "dev_instructions.jsonl", "w") as f:
    for item in dev_instruction:
        f.write(json.dumps(item) + "\n")

with open(processed_dir / "test_instructions.jsonl", "w") as f:
    for item in test_instruction:
        f.write(json.dumps(item) + "\n")
print("Instruction-response data saved as JSONL files")

combined_instructions = train_instruction + dev_instruction
with open(processed_dir / "combined_instructions.jsonl", "w") as f:
    for item in combined_instructions:
        f.write(json.dumps(item) + "\n")
print(f"Combined training data saved ({len(combined_instructions)} examples)")

print(f"\nProcessed data location: {processed_dir}")


Saving cleaned datasets...
Cleaned data saved as parquet files

Saving instruction-response format for LoRA training...
Instruction-response data saved as JSONL files
Combined training data saved (6159 examples)

Processed data location: d:\Education\Level 4\Term 2\NLU\Project\Opinion-Mining-LLM\data\processed


## 7. Data Quality Analysis and Statistics

In [34]:
def analyze_dataset_statistics(df, instruction_data, dataset_name):
    stats = {
        "dataset": dataset_name,
        "total_examples": len(df),
        "total_instruction_pairs": len(instruction_data),
    }
    
    polarity_counts = []
    total_aspects = 0
    
    for row in df.itertuples():
        try:
            output_dict = json.loads(row.output)
            aspects = output_dict.get("aspects", [])
            total_aspects += len(aspects)
            polarity_counts.extend([a.get("polarity") for a in aspects])
        except:
            pass
    
    from collections import Counter
    
    stats["polarity_distribution"] = dict(Counter(polarity_counts))
    stats["avg_aspects_per_review"] = total_aspects / len(df) if len(df) > 0 else 0
    
    review_lengths = []
    for row in df.itertuples():
        try:
            if isinstance(row.input, str):
                review_lengths.append(len(row.input.split()))
        except:
            pass
    
    if review_lengths:
        import statistics
        stats["avg_review_length_words"] = statistics.mean(review_lengths)
        stats["min_review_length"] = min(review_lengths)
        stats["max_review_length"] = max(review_lengths)
    
    return stats

print("=" * 70)
print("DATA QUALITY STATISTICS")
print("=" * 70 + "\n")

train_stats = analyze_dataset_statistics(train_cleaned, train_instruction, "Train")
dev_stats = analyze_dataset_statistics(dev_cleaned, dev_instruction, "Dev")
test_stats = analyze_dataset_statistics(test_cleaned, test_instruction, "Test")

for stats in [train_stats, dev_stats, test_stats]:
    print(f"\n{stats['dataset']} Dataset:")
    print(f"  Total examples: {stats['total_examples']}")
    print(f"  Instruction pairs: {stats['total_instruction_pairs']}")
    print(f"  Avg aspects per review: {stats['avg_aspects_per_review']:.2f}")
    print(f"  Polarity distribution: {stats['polarity_distribution']}")
    print(f"  Avg review length: {stats.get('avg_review_length_words', 0):.1f} words")
    print(f"  Review length range: {stats.get('min_review_length', 0)}-{stats.get('max_review_length', 0)} words")


print("\n" + "=" * 70)
print("OVERALL PREPROCESSING SUMMARY")
print("=" * 70)
total_cleaned = len(train_cleaned) + len(dev_cleaned) + len(test_cleaned)
total_original = train_df.shape[0] + dev_df.shape[0] + test_df.shape[0]
print(f"Total data points processed: {total_original}")
print(f"Total data points retained: {total_cleaned}")
print(f"Data retention rate: {total_cleaned / total_original * 100:.2f}%")
print(f"Total data points removed: {total_original - total_cleaned}")


DATA QUALITY STATISTICS


Train Dataset:
  Total examples: 5959
  Instruction pairs: 5959
  Avg aspects per review: 0.98
  Polarity distribution: {'neutral': 1082, 'positive': 3104, 'negative': 1634}
  Avg review length: 14.0 words
  Review length range: 1-78 words

Dev Dataset:
  Total examples: 200
  Instruction pairs: 200
  Avg aspects per review: 0.72
  Polarity distribution: {'positive': 97, 'negative': 34, 'neutral': 14}
  Avg review length: 11.1 words
  Review length range: 2-35 words

Test Dataset:
  Total examples: 1572
  Instruction pairs: 1572
  Avg aspects per review: 1.11
  Polarity distribution: {'positive': 1060, 'negative': 318, 'neutral': 361}
  Avg review length: 13.1 words
  Review length range: 2-60 words

OVERALL PREPROCESSING SUMMARY
Total data points processed: 7731
Total data points retained: 7731
Data retention rate: 100.00%
Total data points removed: 0


## 8. Sample Verification - Visual Inspection

In [ ]:
print("=" * 70)
print("SAMPLE CLEANED DATA - TRAIN SET")
print("=" * 70 + "\n")

for sample_idx in range(min(3, len(train_cleaned))):
    print(f"\n--- Sample {sample_idx + 1} ---")
    row = train_cleaned.iloc[sample_idx]
    output_json = json.loads(row["output"])
    print(f"Input (Review): {row['input'][:100]}...")
    print(f"Domain: {row.get('domain', 'N/A')}")
    print(f"Aspects found: {len(output_json.get('aspects', []))}")
    for aspect in output_json.get('aspects', [])[:2]:
        print(f"  - Term: {aspect.get('term')} | Polarity: {aspect.get('polarity')}")

print("\n" + "=" * 70)
print("SAMPLE INSTRUCTION-RESPONSE PAIRS")
print("=" * 70 + "\n")

for sample_idx in range(min(2, len(train_instruction))):
    print(f"\n--- Instruction Pair {sample_idx + 1} ---")
    pair = train_instruction[sample_idx]
    print("INSTRUCTION:")
    print(pair["instruction"][:300] + "...")
    print("\nOUTPUT:")
    output_json = json.loads(pair["output"])
    print(json.dumps(output_json, indent=2)[:400] + "...")

print("\n Data preprocessing pipeline complete!")
print("Ready to proceed to: notebooks/02_lora_finetuning.ipynb")


SAMPLE CLEANED DATA - TRAIN SET


--- Sample 1 ---
Input (Review): I charge it at night and skip taking the cord with me because of the good battery life....
Domain: laptops
Aspects found: 2
  - Term: cord | Polarity: neutral
  - Term: battery life | Polarity: positive

--- Sample 2 ---
Input (Review): I bought a HP Pavilion DV41222nr laptop and have had so many problems with the computer....
Domain: laptops
Aspects found: 0

--- Sample 3 ---
Input (Review): The tech guy then said the service center does not do 1to1 exchange and I have to direct my concern ...
Domain: laptops
Aspects found: 3
  - Term: service center | Polarity: negative
  - Term: "sales" team | Polarity: negative

SAMPLE INSTRUCTION-RESPONSE PAIRS


--- Instruction Pair 1 ---
INSTRUCTION:
Extract the aspect terms and their polarity from the text....

OUTPUT:
{
  "aspects": [
    {
      "term": "cord",
      "polarity": "neutral"
    },
    {
      "term": "battery life",
      "polarity": "positive"
    }
  ]
}...

-